# IT594 Project: Constraint-Preserving E-Commerce Search
## Notebook 01: Data loading, NDCG basics, and constraint exploration

**Project in one line:** build a product search system and test whether it finds relevant products
*without* breaking requirements the shopper explicitly asked for (size, color, material), measuring
relevance and requirement violations **separately**.

**What this notebook does**
1. Load the three WANDS files
2. Inspect the data by eye
3. Build NDCG (our relevance metric) from scratch and verify it against sklearn
4. Explore `product_features` to find which attributes can support a constraint study
5. Check which attributes shoppers actually mention in queries
6. Write down findings that will drive the project proposal

## 1. Setup and load data
WANDS files are **tab-separated**, so `sep="\t"` is required.

In [ ]:
import os
import re
import numpy as np
import pandas as pd
from collections import Counter

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

PATH = "WANDS/dataset/"   # change if your WANDS folder is elsewhere
assert os.path.exists(PATH + "product.csv"), f"Not found. Current folder is: {os.getcwd()}"

products = pd.read_csv(PATH + "product.csv", sep="\t")
queries  = pd.read_csv(PATH + "query.csv",   sep="\t")
labels   = pd.read_csv(PATH + "label.csv",   sep="\t")

print("products:", products.shape)
print("queries: ", queries.shape)
print("labels:  ", labels.shape)

products: (42994, 9)
queries:  (480, 3)
labels:   (233448, 4)


Expected: about 42,994 products, 480 queries, and 233,448 labels.

## 2. First look at the data
**product.csv**: one row per product (name, class, description, features).

In [ ]:
products.head(3)

,product_id,product_name,product_class,category hierarchy,product_description,product_features,rating_count,average_rating,review_count
0,0,solid wood platform bed,Beds,Furniture / Bedroom Furniture / Beds & Headboards / Beds / Twin Beds,"good , deep sleep can be quite difficult to have in this busy age . fortunately , there ’ s an antidote to such a pr...",overallwidth-sidetoside:64.7|dsprimaryproductstyle : modern|countryoforigin-additionaldetails : made in viet nam|hea...,15.0,4.5,15.0
1,1,all-clad 7 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances / Pressure & Slow Cookers / Slow Cookers / Slow Slow Cookers,"create delicious slow-cooked meals , from tender meat to flavorful veggies , with this easy-to-use slow cooker . the...",capacityquarts:7|producttype : slow cooker|programmablecookingsettings : slow cook|overallheight-toptobottom:14.2|ov...,100.0,2.0,98.0
2,2,all-clad electrics 6.5 qt . slow cooker,Slow Cookers,Kitchen & Tabletop / Small Kitchen Appliances / Pressure & Slow Cookers / Slow Cookers / Slow Slow Cookers,"prepare home-cooked meals on any schedule with this essential slow cooker , featuring a dishwasher-safe insert and 2...",features : keep warm setting|capacityquarts:6.5|programmablefoodsettings : soup|warrantylength:1 year| : slow cooker...,208.0,3.0,181.0


**query.csv**: what shoppers typed, plus the product type Wayfair assigned (`query_class`).

In [ ]:
queries.sample(15, random_state=1)

,query_id,query,query_class
293,297,seat cushions desk,Chair Accessories
283,287,cover set for outdoor furniture,Furniture Covers
34,34,bar stool with backrest,Bar Stools
380,386,alter furniture,Outdoor Conversation Sets
430,436,full bed with trundle,Kids Beds
363,369,wrought tv stand,TV Stands & Entertainment Centers
383,389,brush nickel faucet bathroom,Bathroom Sink Faucets
66,66,novara pergola,Pergolas
250,254,comfortable accent chair,Accent Chairs
31,31,burnt orange curtains,Curtains & Drapes


**label.csv**: how relevant each product is for each query.
`Exact` = fully matches, `Partial` = matches the type but misses something, `Irrelevant` = does not match.

In [ ]:
print(labels["label"].value_counts())
print()
print(labels["label"].value_counts(normalize=True).round(3))
print("\nJudged products per query (min / median / max):",
      labels.groupby("query_id").size().agg(["min", "median", "max"]).tolist())

label
Partial       146633
Irrelevant     61201
Exact          25614
Name: count, dtype: int64

label
Partial       0.628
Irrelevant    0.262
Exact         0.110
Name: proportion, dtype: float64

Judged products per query (min / median / max): [1.0, 216.5, 4329.0]


> **Note:** not every product is labeled for every query. Only the judged products (a few hundred per
> query) can be scored. This matters for NDCG below.

### Read one query with its labeled products
This is the manual inspection the project asks for. Change `QID` to look at other queries.

In [ ]:
QID = 11   # try 0, 11, 15, ...
print("QUERY:", queries.loc[queries.query_id == QID, "query"].iloc[0])
view = (labels[labels.query_id == QID]
        .merge(products[["product_id", "product_name", "product_class"]], on="product_id"))
for lab in ["Exact", "Partial", "Irrelevant"]:
    print(f"\n--- {lab} (showing 5) ---")
    print(view[view.label == lab][["product_name", "product_class"]].head(5).to_string(index=False))

QUERY: ombre rug

--- Exact (showing 5) ---
                                product_name product_class
     abbott ombre flatweave cotton coral rug     Area Rugs
                   adela ombre blue area rug     Area Rugs
alberich ombre flatweave gray/green area rug     Area Rugs
                  andile ombre blue area rug     Area Rugs
             anran ombre black/gray area rug     Area Rugs

--- Partial (showing 5) ---
                              product_name product_class
             abstract black/ivory area rug     Area Rugs
       aradia braided cotton aqua area rug     Area Rugs
      aradia braided cotton beige area rug     Area Rugs
aradia braided cotton black/white area rug     Area Rugs
       aradia braided cotton blue area rug     Area Rugs

--- Irrelevant (showing 5) ---
                                    product_name         product_class
    carolie ombre semi-sheer rod pocket curtains     Curtains & Drapes
              euphemia ombre glass console table Sofa & C

## 3. NDCG: our relevance metric

**Idea:** a good ranked list puts the most relevant items first.

1. **Gain:** Exact = 2, Partial = 1, Irrelevant = 0
2. **Discount:** an item at position *i* counts less, divided by `log2(i + 1)`
3. **DCG** = sum of gain / discount over the top-k
4. **NDCG** = DCG / ideal DCG, so the score is between 0 and 1

**Our choices (state these in the report):**
- Linear gain (2/1/0)
- Unlabeled products count as 0
- Ideal ranking is built from judged products only

In [ ]:
GAIN = {"Exact": 2, "Partial": 1, "Irrelevant": 0}
labels["gain"] = labels["label"].map(GAIN)
print("Unmapped labels:", labels["gain"].isna().sum(), "(should be 0)")

# qrels[query_id] = {product_id: gain}
qrels = {q: dict(zip(g["product_id"], g["gain"])) for q, g in labels.groupby("query_id")}

def dcg(gains):
    return sum(g / np.log2(i + 2) for i, g in enumerate(gains))

def ndcg_at_k(ranked_ids, judged, k=10):
    """ranked_ids: product ids in the order our system returned them.
       judged: {product_id: gain} for this query."""
    gains = [judged.get(p, 0) for p in ranked_ids[:k]]          # unlabeled = 0
    ideal = sorted(judged.values(), reverse=True)[:k]
    return dcg(gains) / dcg(ideal) if dcg(ideal) > 0 else 0.0

Unmapped labels: 0 (should be 0)


**Sanity check.** System returns Partial, Exact, Irrelevant. Hand calculation earlier gave about 0.86.

In [ ]:
mine = ndcg_at_k(["a", "b", "c"], {"a": 1, "b": 2, "c": 0}, k=3)
print("My NDCG@3:", round(mine, 4))

from sklearn.metrics import ndcg_score
true_gains = np.array([[1, 2, 0]])
scores     = np.array([[3, 2, 1]])        # higher score = ranked higher
sk = ndcg_score(true_gains, scores, k=3)
print("sklearn  :", round(sk, 4))
assert abs(mine - sk) < 1e-9, "Implementations disagree!"
print("Match.")

My NDCG@3: 0.8597
sklearn  : 0.8597
Match.


**Two reference points.** A perfect ranking scores 1.0. A random ranking gives a floor that real systems must beat.

In [ ]:
rng = np.random.default_rng(0)
perfect, rand = [], []
for q, judged in qrels.items():
    ids = list(judged.keys())
    perfect.append(ndcg_at_k(sorted(ids, key=lambda p: -judged[p]), judged))
    rand.append(ndcg_at_k(list(rng.permutation(ids)), judged))
print("Perfect ranking mean NDCG@10:", round(np.mean(perfect), 3))
print("Random ranking  mean NDCG@10:", round(np.mean(rand), 3), "  <- baselines must beat this")

Perfect ranking mean NDCG@10: 0.998
Random ranking  mean NDCG@10: 0.621   <- baselines must beat this


## 4. Explore `product_features`: which constraints can the data support?

Each product has one text string like `color : navy | primarymaterial : wool`.
We turn it into a dictionary so we can check, for example, "is color = navy?"

In [ ]:
for i in [0, 100, 500]:
    print(products.loc[i, "product_name"])
    print(products.loc[i, "product_features"][:300])
    print("-----")

solid wood platform bed
overallwidth-sidetoside:64.7|dsprimaryproductstyle : modern|countryoforigin-additionaldetails : made in viet nam|headboardtype : slat|headboardheight-toptobedframe:25.35|weightcapacity:500|estimatedtimetosetup:120|overalllength-headtotoe:77.9|levelofassembly : full assembly needed|overallproductweig
-----
telemark 4 shelf etagere bookcase
additionaltoolsrequirednotincluded : screw driver|numberofshelftiers:4|producttype : etagere|backpanel : open|dswoodtone : gray wood|levelofassembly : full assembly needed|supplierintendedandapproveduse : non residential use|shelfdepth-fronttoback:13.7|framematerial : solid + manufactured wood|overa
-----
larrabee solid wood sleigh bed
overallproductweight:90|overalllength-headtotoe:93|style : traditional|framematerial : solid wood|bedstorageincluded : none|overallheight-toptobottom:48|headboardheight-toptobedframe:31|weightcapacity:320|overallwidth-sidetoside:57|baseconstruction : standard|additionaltoolsrequirednotincluded : 

In [ ]:
def parse_features(s):
    """'color : navy|material:wool' -> {'color': 'navy', 'material': 'wool'}"""
    d = {}
    if pd.isna(s):
        return d
    for pair in s.split("|"):
        if ":" in pair:
            k, v = pair.split(":", 1)
            d[k.strip().lower()] = v.strip()
    return d

products["feat"] = products["product_features"].apply(parse_features)
print("Products with no features at all:", (products["feat"].apply(len) == 0).sum())

Products with no features at all: 0


### 4a. Most common attributes across all products

In [ ]:
counter = Counter()
for d in products["feat"]:
    counter.update(d.keys())

top = pd.DataFrame(counter.most_common(40), columns=["attribute", "num_products"])
top["percent"] = (top["num_products"] / len(products) * 100).round(1)
top

,attribute,num_products,percent
0,overallproductweight,39810,92.6
1,,34990,81.4
2,countryoforigin,34017,79.1
3,supplierintendedandapproveduse,32507,75.6
4,overallheight-toptobottom,31636,73.6
5,overallwidth-sidetoside,31401,73.0
6,commercialwarranty,30517,71.0
7,productwarranty,29184,67.9
8,overalldepth-fronttoback,27728,64.5
9,purposefuldistressingtype,26320,61.2


Attributes under about 10 to 20 percent of products are too sparse to rely on overall.
Notice that "material" is split across many fields (`primarymaterial`, `framematerial`, `material`, ...).
This is the kind of **inconsistent metadata** the project wants you to study.

In [ ]:
# All attribute names that mention a constraint-like word
for word in ["color", "material", "size", "width", "length", "height", "brand"]:
    hits = [(k, v) for k, v in counter.items() if word in k]
    hits.sort(key=lambda x: -x[1])
    print(f"{word:9s} -> {len(hits)} attribute names; top: {hits[:5]}")

color     -> 121 attribute names; top: [('color', 15214), ('basecolor', 5339), ('topcolor', 4155), ('legcolor', 3540), ('upholsterycolor', 2605)]
material  -> 364 attribute names; top: [('primarymaterial', 10376), ('framematerial', 9519), ('material', 8253), ('upholsterymaterial', 6496), ('basematerial', 6420)]
size      -> 94 attribute names; top: [('mattresssize', 1790), ('setsize', 678), ('size', 673), ('rugsize', 624), ('multipledrawersizes', 598)]
width     -> 326 attribute names; top: [('overallwidth-sidetoside', 31401), ('seatwidth-sidetoside', 4824), ('minimumdoorwidth-sidetoside', 2873), ('shelfwidth-sidetoside', 2213), ('drawerinteriorwidth-sidetoside', 1045)]
length    -> 170 attribute names; top: [('warrantylength', 21062), ('overalllength-endtoend', 5428), ('overalllength-headtotoe', 2901), ('individualtilelength', 836), ('adjustablehanginglength', 776)]
height    -> 380 attribute names; top: [('overallheight-toptobottom', 31636), ('seatheight-floortoseat', 5150), ('adjust

### 4b. Per-category coverage
A constraint study works best in categories where the attribute is **usually present**.

In [ ]:
def coverage(df, attrs):
    """percent of products in df that have each attribute"""
    return {a: round(100 * df["feat"].apply(lambda d: a in d).mean(), 1) for a in attrs}

ATTRS = ["color", "primarycolor", "material", "primarymaterial", "style"]
big_classes = products["product_class"].value_counts().head(15).index

rows = []
for cls in big_classes:
    sub = products[products["product_class"] == cls]
    rows.append({"product_class": cls, "n_products": len(sub), **coverage(sub, ATTRS)})
pd.DataFrame(rows)

,product_class,n_products,color,primarycolor,material,primarymaterial,style
0,Area Rugs,1316,0.0,99.8,98.6,0.0,51.7
1,Accent Chairs,1169,0.0,0.0,0.0,0.0,42.1
2,Beds,1112,85.9,0.0,0.0,0.0,45.1
3,Coffee & Cocktail Tables,1085,0.0,0.0,0.0,0.0,56.5
4,Floor & Wall Tile,861,81.8,0.0,97.6,0.0,0.0
5,Bar Stools,775,0.0,0.0,0.0,0.0,53.5
6,Accent Pillows,698,76.5,0.0,0.0,0.0,45.1
7,End Tables,607,0.0,0.0,0.0,0.0,54.2
8,Desks,595,0.0,0.0,0.0,0.0,43.9
9,Dining Tables,591,0.0,0.0,0.0,0.0,66.5


### 4c. Zoom in: Area Rugs (size is a classic constraint)

In [ ]:
rugs = products[products["product_class"] == "Area Rugs"]
print("Rugs:", len(rugs))

c = Counter()
for d in rugs["feat"]:
    c.update(d.keys())
rug_attrs = pd.DataFrame(c.most_common(20), columns=["attribute", "num_rugs"])
rug_attrs["percent_of_rugs"] = (rug_attrs["num_rugs"] / len(rugs) * 100).round(1)
rug_attrs

Rugs: 1316


,attribute,num_rugs,percent_of_rugs
0,primarycolor,1313,99.8
1,overalllength-endtoend,1310,99.5
2,rugsets,1310,99.5
3,fringetassel,1307,99.3
4,overallwidth-sidetoside,1305,99.2
5,overallproductweight,1302,98.9
6,one-of-a-kind,1302,98.9
7,location,1299,98.7
8,material,1298,98.6
9,purposefuldistressingtype,1293,98.3


In [ ]:
# IMPORTANT: rugs do NOT use the attribute name "color". They use "primarycolor".
# Different categories name the same idea differently (a metadata inconsistency).
for name in ["color", "primarycolor"]:
    print(f"{name:13s}: {rugs['feat'].apply(lambda d: name in d).sum()} of {len(rugs)} rugs")

rug_colors = rugs["feat"].apply(lambda d: d.get("primarycolor")).dropna().str.lower()
print("\nMost common rug color values:")
print(rug_colors.value_counts().head(25))

color        : 0 of 1316 rugs
primarycolor : 1313 of 1316 rugs

Most common rug color values:
feat
gray             92
teal             72
red              45
blue             45
plum             44
beige            30
multi-colored    26
dark gray        22
purple           18
ivory            18
green            17
brown            16
pink             14
black            13
beige/black      13
red/beige        11
magenta           9
cream             9
yellow            9
plum/ivory        9
brown/beige       9
red/gray          8
black/beige       8
multi             8
multi-color       7
Name: count, dtype: int64


In [ ]:
# Rug size is messy: a text field (rugsize) on under half the rugs,
# plus numeric length/width fields on almost all of them.
print("rugsize present      :", rugs["feat"].apply(lambda d: "rugsize" in d).sum(), "of", len(rugs))
print("length+width present :", rugs["feat"].apply(lambda d: "overalllength-endtoend" in d and "overallwidth-sidetoside" in d).sum(), "of", len(rugs))
print("\nSample rugsize strings:")
print(rugs["feat"].apply(lambda d: d.get("rugsize")).dropna().sample(8, random_state=1).tolist())
print("\nSample numeric length / width:")
print(rugs["feat"].apply(lambda d: (d.get("overalllength-endtoend"), d.get("overallwidth-sidetoside"))).head(8).tolist())

rugsize present      : 603 of 1316
length+width present : 1304 of 1316

Sample rugsize strings:
["7 ' 6 '' x 9 ' 6 ''", "rectangle 7'10 '' x 10 ' 3 ''", "3 ' 3 '' x 6 ' 8 ''", "9 ' x 12'", '', "square 7'10 ''", "round 6 ' 7 ''", '42x42']

Sample numeric length / width:
[('30', '36'), ('72', '72'), ('48', '96'), ('72', '72'), ('28', '36'), ('120', '60'), ('60', '60'), ('30', '72')]


**Three things to notice (all feed your error analysis):**
1. **Same idea, different attribute name.** Rugs use `primarycolor`, other classes use `color`. A constraint checker must know all the names.
2. **Overlapping values.** `gray` vs `dark gray`, `multi-colored`, `teal` vs `blue`. Deciding when a rug "satisfies" a navy request is the **ambiguity** problem in the brief.
3. **Size is inconsistent.** Text like `9 ' x 12 ' 3 ''` needs normalizing, or you can use the numeric length/width fields instead.

## 5. Do shoppers actually ask for these attributes?
A constraint is only useful if queries mention it. We scan the 480 queries for simple patterns.
This is a rough first pass, not a final parser.

In [ ]:
COLORS = ["black","white","blue","navy","gray","grey","red","green","beige","brown","pink","yellow",
          "gold","silver","ivory","cream","purple","orange","teal","turquoise"]
MATERIALS = ["wood","wooden","leather","metal","glass","marble","velvet","fabric","linen","rattan",
             "wicker","bamboo","steel","iron","brass","cotton","wool","acrylic","ceramic"]

def has_word(q, words):
    toks = set(re.findall(r"[a-z]+", q.lower()))
    return any(w in toks for w in words)

SIZE_PATTERN = re.compile(r"\b\d+\s*(x|\')\s*\d+|\b\d+\s*(ft|foot|feet|inch|in|\"|cm)\b|\b(king|queen|twin|full)\b|\b(small|large|oversized)\b", re.I)

queries["has_color"]    = queries["query"].apply(lambda q: has_word(q, COLORS))
queries["has_material"] = queries["query"].apply(lambda q: has_word(q, MATERIALS))
queries["has_size"]     = queries["query"].apply(lambda q: bool(SIZE_PATTERN.search(q)))
queries["any_constraint"] = queries[["has_color", "has_material", "has_size"]].any(axis=1)

print(queries[["has_color", "has_material", "has_size", "any_constraint"]].sum())
print(f"\n{queries['any_constraint'].mean():.0%} of queries mention at least one candidate constraint")

has_color          42
has_material       34
has_size           39
any_constraint    105
dtype: int64

22% of queries mention at least one candidate constraint


In [ ]:
queries[queries.has_color].head(15)

,query_id,query,query_class,has_color,has_material,has_size,any_constraint
3,3,turquoise pillows,Accent Pillows,True,False,False,True
15,15,black 5 drawer dresser by guilford,Dressers & Chests,True,False,False,True
22,22,light and navy blue decorative pillow,Accent Pillows,True,False,False,True
23,23,stoneford end tables white and wood,End Tables,True,True,False,True
31,31,burnt orange curtains,Curtains & Drapes,True,False,False,True
32,32,dark gray dresser,Dressers & Chests,True,False,False,True
46,46,beach blue headboard,Headboards,True,False,False,True
49,49,moen matte black hooks,Towel & Robe Hooks,True,False,False,True
50,50,tufted chair with gold legs,Accent Chairs,True,False,False,True
55,55,tollette teal outdoor rug,Area Rugs,True,False,False,True


In [ ]:
queries[queries.has_size].head(15)

,query_id,query,query_class,has_color,has_material,has_size,any_constraint
10,10,king poster bed,Beds,False,False,True,True
12,12,large spoon and fork wall decor,Wall Décor,False,False,True,True
16,16,blk 18x18 seat cushions,Furniture Cushions,False,False,True,True
30,30,3 1/2 inch drawer pull,Cabinet and Drawer Pulls,False,False,True,True
76,76,46 inch closet door,Interior Doors,False,False,True,True
84,84,full metal bed rose gold,Beds,True,True,True,True
101,101,48 in entry table with side by side drawer,Sofa & Console Tables,False,False,True,True
122,122,king size bed,Beds,False,False,True,True
143,143,queen wingback chair,Accent Chairs,False,False,True,True
170,171,nectar queen mattress,Foam and Latex Mattresses,False,False,True,True


### Which product classes have the most constrained queries?

In [ ]:
by_class = (queries.groupby("query_class")
            .agg(n_queries=("query_id", "size"),
                 n_color=("has_color", "sum"),
                 n_material=("has_material", "sum"),
                 n_size=("has_size", "sum"))
            .sort_values("n_queries", ascending=False))
by_class.head(15)

,n_queries,n_color,n_material,n_size
query_class,,,,
Wall Art,20,0,0,0
Accent Chairs,16,3,3,2
Beds,15,2,2,5
Area Rugs,15,4,1,1
Dressers & Chests,10,4,0,0
Coffee & Cocktail Tables,10,1,1,0
Desks,9,0,0,0
Accent Pillows,8,4,0,0
End Tables,8,1,2,0


## 6. Findings (fill this in, it feeds the one-page proposal)

**Attributes common enough to study (3 to 5):**
- Color: 35% of products (color); rugs use primarycolor (99.8%)
- Material: primarymaterial, framematerial, material (each about 8,000 to 10,000 products)
- Dimensions: width, height, length (numeric, on up to 73% of products)
- Style: common but too vague to be a hard constraint

**Categories with the most complete attributes:**
- Area Rugs: color 99.8%, material 98.6%, size 99%
- Wall Art: color 97.6%
- Floor & Wall Tile: color 82%, material 98%
- Beds: color 86%
- Accent Pillows: color 77%

**Number of queries mentioning each attribute:**
- color:42
- material:34
- size:39
- Any of the three:105(22%)

**5 examples where relevance and a requirement could disagree** (query, product, why):
1."gray dresser": an Exact product is black
2."beach blue headboard": an Exact product is taupe
3."black 5 drawer dresser": an Exact product is beige
4."dark gray dresser": an Exact product has color "platinum" (ambiguous)
5."turquoise pillows": an Exact product has a material description in its color field (bad metadata)

**Candidate falsifiable hypothesis:**
- On queries that name a color, dense retrieval (SBERT) has a higher top-10 color-violation rate than BM25, even when its NDCG@10 is equal or higher. It fails if dense is the same or better.

**Open questions / data problems noticed (missing values, inconsistent names, ambiguous values):**
- Same attribute has different names (color, primarycolor, basecolor)
- About half of labeled products have no color value
- Ambiguous values (gray vs dark gray vs platinum, beige/black)
- Wrong values (a material in the color field)
- Messy rug sizes (9 ' x 12 ' 3 '')
- Only about 4 color queries per category, so combine categories